# YOLO v1 — Google Colab T4 학습 노트북

### 실행 순서
1. **런타임 → 런타임 유형 변경 → T4 GPU** 선택 확인
2. 셀을 위에서부터 순서대로 실행
3. VOC 데이터 준비 (1회만 실행, 이후 세션에서는 Drive 마운트 후 스킵)


## 0. GPU 확인

In [ ]:
import torch
print('PyTorch 버전:', torch.__version__)
print('CUDA 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 1. Google Drive 마운트


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Drive 내 작업 폴더 경로 (원하는 위치로 변경 가능) ──
DRIVE_ROOT    = '/content/drive/MyDrive/yolov1'
CKPT_DIR      = f'{DRIVE_ROOT}/checkpoints'
VOC_DIR       = f'{DRIVE_ROOT}/VOC_Detection'   # 데이터셋을 Drive에 올려두는 경우

os.makedirs(CKPT_DIR, exist_ok=True)
print('체크포인트 저장 경로:', CKPT_DIR)

## 2. 패키지 설치

코랩 기본 PyTorch로 동작합니다. 버전 충돌이 없으면 재설치 불필요.

In [ ]:
!pip install -q tqdm pillow

## 3. 소스코드 업로드

In [ ]:
# ── 방법 A: Drive에 소스코드를 올려둔 경우 ──────────────────
# DRIVE_CODE = f'{DRIVE_ROOT}/src'
# !cp {DRIVE_CODE}/*.py /content/

# ── 방법 B: 직접 업로드한 경우 → 이 셀 스킵 ─────────────────
import os
files = ['model.py','dataset.py','transforms.py','loss.py','train.py','evaluate.py']
missing = [f for f in files if not os.path.exists(f'/content/{f}')]
if missing:
    print('⚠️  아직 업로드되지 않은 파일:', missing)
    print('   왼쪽 파일 탭에서 /content/ 에 업로드해주세요.')
else:
    print('✅ 소스코드 모두 확인됨')

## 4. VOC 데이터셋 준비

**처음 한 번만 실행** 이후 세션에서는 Drive에 저장된 데이터를 그대로 사용.

> 데이터가 이미 Drive에 있다면 이 섹션 전체를 스킵

In [ ]:

VOC_DOWNLOAD = '/content/voc_raw'os.makedirs(VOC_DOWNLOAD, exist_ok=True)
%cd {VOC_DOWNLOAD}

!wget -q http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar
!wget -q http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtest_06-Nov-2007.tar
!wget -q http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar

!tar xf VOCtrainval_06-Nov-2007.tar
!tar xf VOCtest_06-Nov-2007.tar
!tar xf VOCtrainval_11-May-2012.tar
print('압축 해제 완료')

In [ ]:
# VOC 데이터를 학습/테스트 구조로 정리하고 CSV 어노테이션 생성
# (원본 레포의 organize_voc.sh + simplify_voc_targets.py 역할)
import shutil, xml.etree.ElementTree as ET
from pathlib import Path

VOC_OUT = VOC_DIR   # Drive에 저장
CLASSES = [
    'person',
    'bird','cat','cow','dog','horse','sheep',
    'aeroplane','bicycle','boat','bus','car','motorbike','train',
    'bottle','chair','diningtable','pottedplant','sofa','tvmonitor'
]

def parse_voc_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    rows = []
    for obj in root.findall('object'):
        name = obj.find('name').text
        if name not in CLASSES:
            continue
        diff = obj.find('difficult')
        if diff is not None and int(diff.text) == 1:
            continue   # difficult 객체 제외
        bb = obj.find('bndbox')
        xmin = int(float(bb.find('xmin').text))
        ymin = int(float(bb.find('ymin').text))
        xmax = int(float(bb.find('xmax').text))
        ymax = int(float(bb.find('ymax').text))
        rows.append((name, xmin, ymin, xmax, ymax))
    return rows

def organize_split(voc_roots, image_sets, out_split):
    img_dir  = Path(VOC_OUT) / out_split / 'images'
    tgt_dir  = Path(VOC_OUT) / out_split / 'targets'
    img_dir.mkdir(parents=True, exist_ok=True)
    tgt_dir.mkdir(parents=True, exist_ok=True)

    count = 0
    for voc_root, split_file in zip(voc_roots, image_sets):
        voc_root = Path(voc_root)
        with open(split_file) as f:
            ids = [l.strip() for l in f if l.strip()]
        for pid in ids:
            src_img = voc_root / 'JPEGImages' / f'{pid}.jpg'
            src_xml = voc_root / 'Annotations' / f'{pid}.xml'
            if not src_img.exists() or not src_xml.exists():
                continue
            rows = parse_voc_xml(src_xml)
            if not rows:
                continue
            # 파일명 충돌 방지: 연도 접두사 추가
            year = voc_root.parent.name[-4:]
            uid = f'{year}_{pid}'
            shutil.copy(src_img, img_dir / f'{uid}.jpg')
            with open(tgt_dir / f'{uid}.csv', 'w') as f:
                f.write('class,xmin,ymin,xmax,ymax\n')
                for r in rows:
                    f.write(','.join(str(x) for x in r) + '\n')
            count += 1
    print(f'  {out_split}: {count}개 이미지 처리 완료')

RAW = Path(VOC_DOWNLOAD) / 'VOCdevkit'

# 학습셋: VOC2007 trainval + VOC2012 trainval
print('학습셋 구성 중...')
organize_split(
    [RAW/'VOC2007', RAW/'VOC2012'],
    [RAW/'VOC2007'/'ImageSets'/'Main'/'trainval.txt',
     RAW/'VOC2012'/'ImageSets'/'Main'/'trainval.txt'],
    'train'
)

# 테스트셋: VOC2007 test
print('테스트셋 구성 중...')
organize_split(
    [RAW/'VOC2007'],
    [RAW/'VOC2007'/'ImageSets'/'Main'/'test.txt'],
    'test'
)

print('\n✅ 데이터셋 준비 완료:', VOC_OUT)

## 5. 사전학습(Pretrain) 가중치 준비

**둘 중 하나만 실행**

In [ ]:
# ── 방법 A: 이미 pretrained_model_weights.pt 파일이 있는 경우 ──
# Drive에 파일을 올려두고 경로만 확인
PRETRAINED_WEIGHTS = f'{CKPT_DIR}/pretrained_model_weights.pt'
import os
if os.path.exists(PRETRAINED_WEIGHTS):
    print('✅ 사전학습 가중치 확인:', PRETRAINED_WEIGHTS)
else:
    print('⚠️  파일 없음. 방법 B(처음부터 학습) 셀을 실행하거나 Drive에 파일을 업로드하세요.')

In [ ]:
# ── 방법 B: 사전학습 가중치 없이 처음부터 탐지 학습 ──────────────
# train.py의 LOAD_MODEL = None 설정과 함께 사용
# 이 셀은 방법 A가 실패했을 때만 실행
PRETRAINED_WEIGHTS = None
print('사전학습 가중치 없이 진행합니다. (LOAD_MODEL = None 모드)')

## 6. 학습 설정 (T4 최적화)

In [ ]:
# train.py를 직접 수정하지 않고, 설정값을 환경변수로 주입

import os

# ── 경로 ────────────────────────────────────────────────────
os.environ['VOC_DIR']        = VOC_DIR
os.environ['CKPT_DIR']       = CKPT_DIR
os.environ['PRETRAINED_W']   = PRETRAINED_WEIGHTS or ''

# ── T4 메모리 최적화 설정 출력 ─────────────────────────────
print('=== T4 최적화 설정 ===')
print('실제 미니배치 크기 : 4장  (BATCH=64, SUBDIVISIONS=16)')
print('  → T4 VRAM 15GB 기준 안전한 크기')
print('  → 메모리 부족 시 SUBDIVISIONS=32로 변경 (미니배치 2장)')
print('DataLoader workers : 2')
print('VOC 데이터 경로    :', VOC_DIR)
print('체크포인트 경로    :', CKPT_DIR)

## 7. 탐지 모델 학습

train.py를 코랩 환경에 맞게 설정값을 오버라이드하여 실행합니다.

In [ ]:
# train.py의 설정값을 코랩용으로 오버라이드 후 실행
# 원본 train.py 코드는 전혀 수정하지 않음

import importlib, sys
sys.path.insert(0, '/content')

# ── 코랩 환경 설정값 ─────────────────────────────────────────
LOAD_MODEL_MODE = 'pretrain' if PRETRAINED_WEIGHTS else None

# train 모듈을 import하고 설정값만 교체
import train as train_module

train_module.PASCAL_VOC_DIR_PATH      = VOC_DIR
train_module.PRETRAINED_MODEL_WEIGHTS = PRETRAINED_WEIGHTS or ''
train_module.TRAINING_CHECKPOINT_PATH = f'{CKPT_DIR}/training_checkpoint.pt'
train_module.TRAINED_MODEL_WEIGHTS    = f'{CKPT_DIR}/trained_model_weights.pt'
train_module.LOAD_MODEL               = LOAD_MODEL_MODE

# T4 메모리 최적화
train_module.BATCH        = 64
train_module.SUBDIVISIONS = 16   # 실제 미니배치 = 4장 (부족하면 32로)
train_module.NUM_WORKERS  = 2    # 코랩 권장값

# ── 학습 시작 ────────────────────────────────────────────────
print(f'학습 모드: LOAD_MODEL = {LOAD_MODEL_MODE}')
print(f'실제 미니배치: {train_module.BATCH // train_module.SUBDIVISIONS}장')
print('학습 시작...')

train_loader, test_loader, model, optimizer, scheduler, criterion = train_module.setup_train()
epoch, mini_batch, train_loss_hist, test_loss_hist = train_module.init_train(model, optimizer, scheduler)
train_module.train(
    train_loader, test_loader, model, optimizer, criterion, scheduler,
    epoch, mini_batch, train_loss_hist, test_loss_hist
)

## 8. 학습 재개 (세션 끊긴 후)

세션이 끊기면 1~3번 셀만 다시 실행한 후 아래 셀을 실행

In [ ]:
import importlib, sys
sys.path.insert(0, '/content')
import train as train_module

# 체크포인트에서 재개
train_module.PASCAL_VOC_DIR_PATH      = VOC_DIR
train_module.TRAINING_CHECKPOINT_PATH = f'{CKPT_DIR}/training_checkpoint.pt'
train_module.TRAINED_MODEL_WEIGHTS    = f'{CKPT_DIR}/trained_model_weights.pt'
train_module.LOAD_MODEL               = 'train'  # 체크포인트에서 재개
train_module.BATCH        = 64
train_module.SUBDIVISIONS = 16
train_module.NUM_WORKERS  = 2

print('체크포인트에서 재개...')
train_loader, test_loader, model, optimizer, scheduler, criterion = train_module.setup_train()
epoch, mini_batch, train_loss_hist, test_loss_hist = train_module.init_train(model, optimizer, scheduler)
print(f'재개 에폭: {epoch} / {train_module.MAX_EPOCHS}')

train_module.train(
    train_loader, test_loader, model, optimizer, criterion, scheduler,
    epoch, mini_batch, train_loss_hist, test_loss_hist
)

## 9. 손실 그래프 확인

In [ ]:
import torch as th
import matplotlib.pyplot as plt

ckpt = th.load(f'{CKPT_DIR}/training_checkpoint.pt', map_location='cpu')
train_hist = ckpt['train_loss_history']
test_hist  = ckpt['test_loss_history']

plt.figure(figsize=(10, 4))
plt.plot(train_hist, label='Train Loss')
plt.plot(test_hist,  label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('YOLO Loss')
plt.title('Training Loss History')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'현재 에폭: {ckpt["epoch"]} / 156')
print(f'최근 Train Loss: {train_hist[-1]:.4f}')
print(f'최근 Test  Loss: {test_hist[-1]:.4f}')

## 10. mAP 평가

In [ ]:
import sys
sys.path.insert(0, '/content')
import evaluate as eval_module

eval_module.PASCAL_VOC_DIR_PATH   = VOC_DIR
eval_module.TRAINED_MODEL_WEIGHTS = f'{CKPT_DIR}/trained_model_weights.pt'
eval_module.NUM_WORKERS = 2
eval_module.PLOT = True

model, test_loader = eval_module.setup_evaluation()
mAP, average_precisions = eval_module.evaluate_model(model, test_loader)

print(f'\n=== 평가 결과 ===')
print(f'Mean Average Precision: {mAP:.1f}%')
print(f'(논문 기준: 63.4%, 이 구현: 63.6%)')

eval_module.plot_class_ap(average_precisions)

### Drive vs 로컬 속도 비교
Drive는 읽기 속도가 느려서 학습이 오래 걸릴 수 있음, 
데이터셋을 `/content/`에 복사하면 빠르지만 세션 끊기면 다시 복사해야 함.

```python
# 빠른 학습을 원할 때: Drive → 로컬 복사 (약 10분 소요)
!cp -r {VOC_DIR} /content/VOC_Detection
VOC_DIR = '/content/VOC_Detection'
```